# 02 — DCGAN sur CIFAR-10

Ce notebook est conçu pour tourner sur **Google Colab avec GPU**.

**Ce qu'on construit :**
- Le Générateur : transforme du bruit en image
- Le Discriminateur : distingue vraie image de fausse
- La boucle d'entraînement
- La visualisation de la progression epoch par epoch

## 1 — Vérification du GPU

In [ ]:
import torch

# Sur Colab : Exécution -> Modifier le type d'exécution -> GPU T4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if device.type == 'cpu':
    print('ATTENTION : GPU non detecte. Va dans Execution -> Modifier le type execution -> GPU')
else:
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'Memoire : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')

## 2 — Imports et hyperparamètres

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import datasets
import matplotlib.pyplot as plt
import numpy as np
import os

# Hyperparametres
Z_DIM        = 100
IMG_CHANNELS = 3
IMG_SIZE     = 32
G_FEATURES   = 64
D_FEATURES   = 64
BATCH_SIZE   = 128
NUM_EPOCHS   = 100
LR           = 0.0002
BETA1        = 0.5
SAVE_EVERY   = 10

# Dossiers de sortie
OUTPUT_DIR = '/content/dcgan_cifar10'
IMGS_DIR   = os.path.join(OUTPUT_DIR, 'progression')
os.makedirs(IMGS_DIR, exist_ok=True)

print('Configuration OK')
print(f'  BATCH_SIZE : {BATCH_SIZE}')
print(f'  NUM_EPOCHS : {NUM_EPOCHS}')
print(f'  SAVE_EVERY : {SAVE_EVERY} epochs')

## 3 — Chargement de CIFAR-10

CIFAR-10 = 50 000 images 32x32 en 10 classes.  
On normalise vers **[-1, 1]** car le generateur utilise `tanh` en sortie.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# download=True telecharge CIFAR-10 automatiquement (~170 Mo)
dataset = datasets.CIFAR10(
    root='/content',
    train=True,
    download=True,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f'Dataset      : {len(dataset)} images')
print(f'Batches/epoch: {len(dataloader)}')

imgs, _ = next(iter(dataloader))
print(f'Shape batch  : {imgs.shape}')
print(f'Valeurs      : min={imgs.min():.2f}  max={imgs.max():.2f}  -> dans [-1, 1]')

## 4 — Générateur

Transforme z (100 nombres) en image 32x32 par upsampling progressif :
```
z(100,1,1) -> (512,4,4) -> (256,8,8) -> (128,16,16) -> (3,32,32)
```
**ConvTranspose2d** : agrandit la taille spatiale a chaque couche.  
**BatchNorm** : stabilise l'entrainement.  
**Tanh** : force la sortie dans [-1, 1].

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim, features, channels):
        super().__init__()
        self.net = nn.Sequential(
            # z(100,1,1) -> (512, 4, 4)
            nn.ConvTranspose2d(z_dim, features * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features * 8),
            nn.ReLU(True),
            # (512, 4, 4) -> (256, 8, 8)
            nn.ConvTranspose2d(features * 8, features * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 4),
            nn.ReLU(True),
            # (256, 8, 8) -> (128, 16, 16)
            nn.ConvTranspose2d(features * 4, features * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.ReLU(True),
            # (128, 16, 16) -> (3, 32, 32)
            nn.ConvTranspose2d(features * 2, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


G = Generator(Z_DIM, G_FEATURES, IMG_CHANNELS).to(device)
z_test = torch.randn(4, Z_DIM, 1, 1).to(device)
print(f'Entree G : {z_test.shape}')
print(f'Sortie G : {G(z_test).shape}')
print(f'Parametres G : {sum(p.numel() for p in G.parameters()):,}')

## 5 — Discriminateur

Reduit l'image a un score unique :
```
(3,32,32) -> (64,16,16) -> (128,8,8) -> (256,4,4) -> score[0,1]
```
**LeakyReLU(0.2)** : laisse passer 20% des valeurs negatives.  
**Sigmoid** : convertit en probabilite [0, 1].

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels, features):
        super().__init__()
        self.net = nn.Sequential(
            # (3,32,32) -> (64,16,16) — pas de BatchNorm sur la 1ere couche
            nn.Conv2d(channels, features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # (64,16,16) -> (128,8,8)
            nn.Conv2d(features, features * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # (128,8,8) -> (256,4,4)
            nn.Conv2d(features * 2, features * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # (256,4,4) -> (1,1,1)
            nn.Conv2d(features * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.net(img).view(-1)


D = Discriminator(IMG_CHANNELS, D_FEATURES).to(device)
img_test = torch.randn(4, IMG_CHANNELS, IMG_SIZE, IMG_SIZE).to(device)
print(f'Entree D : {img_test.shape}')
print(f'Sortie D : {D(img_test).shape}')
print(f'Parametres D : {sum(p.numel() for p in D.parameters()):,}')

## 6 — Initialisation, perte, optimiseurs

In [ ]:
def init_weights(model):
    """Initialise les poids selon les recommandations du papier DCGAN."""
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)

G.apply(init_weights)
D.apply(init_weights)

criterion   = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

# Vecteur fixe pour suivre la progression — on utilisera toujours ce meme z
z_fixed = torch.randn(64, Z_DIM, 1, 1).to(device)

print('Poids initialises, optimiseurs prets.')

## 7 — Boucle d'entraînement

A chaque iteration :
1. **Entrainer D** : vraies images -> cible 1, fausses -> cible 0
2. **Entrainer G** : veut que D sorte 1 sur ses images generees

In [ ]:
def save_snapshot(generator, z, epoch):
    generator.eval()
    with torch.no_grad():
        imgs = (generator(z).cpu() * 0.5 + 0.5).clamp(0, 1)
    fig, axes = plt.subplots(8, 8, figsize=(10, 10))
    fig.suptitle(f'Epoch {epoch}', fontsize=12)
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i].permute(1, 2, 0).numpy())
        ax.axis('off')
    plt.tight_layout()
    path = os.path.join(IMGS_DIR, f'epoch_{epoch:04d}.png')
    plt.savefig(path, dpi=80, bbox_inches='tight')
    plt.close()
    generator.train()
    return path


losses_G        = []
losses_D        = []
saved_snapshots = []

print(f'Debut entrainement — {NUM_EPOCHS} epochs | {len(dataloader)} batches/epoch')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    sum_G = 0
    sum_D = 0

    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        n         = real_imgs.size(0)
        real_lbl  = torch.ones(n,  device=device)
        fake_lbl  = torch.zeros(n, device=device)

        # Etape 1 : entrainer D
        optimizer_D.zero_grad()
        loss_real = criterion(D(real_imgs), real_lbl)
        z         = torch.randn(n, Z_DIM, 1, 1, device=device)
        fake_imgs = G(z)
        loss_fake = criterion(D(fake_imgs.detach()), fake_lbl)
        loss_D    = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # Etape 2 : entrainer G
        optimizer_G.zero_grad()
        loss_G = criterion(D(fake_imgs), real_lbl)
        loss_G.backward()
        optimizer_G.step()

        sum_G += loss_G.item()
        sum_D += loss_D.item()

    avg_G = sum_G / len(dataloader)
    avg_D = sum_D / len(dataloader)
    losses_G.append(avg_G)
    losses_D.append(avg_D)

    if epoch % SAVE_EVERY == 0 or epoch == 1:
        path = save_snapshot(G, z_fixed, epoch)
        saved_snapshots.append((epoch, path))
        print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}] G={avg_G:.4f} D={avg_D:.4f} | snapshot')
    elif epoch % 5 == 0:
        print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}] G={avg_G:.4f} D={avg_D:.4f}')

print('-' * 65)
print('Termine.')

## 8 — Visualisation des résultats

In [ ]:
# Courbes de perte
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses_G, label='Generateur',     color='royalblue')
ax.plot(losses_D, label='Discriminateur', color='tomato')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Evolution des pertes — DCGAN CIFAR-10')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'losses.png'), dpi=100)
plt.show()

In [ ]:
from matplotlib.image import imread

# Progression complete : tous les snapshots cote a cote
n = len(saved_snapshots)
fig, axes = plt.subplots(1, n, figsize=(n * 3, 3))
fig.suptitle('Progression DCGAN CIFAR-10', fontsize=13)

for i, (epoch, path) in enumerate(saved_snapshots):
    ax = axes[i] if n > 1 else axes
    ax.imshow(imread(path))
    ax.set_title(f'Epoch {epoch}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'progression.png'), dpi=100, bbox_inches='tight')
plt.show()

## 9 — Sauvegarde et téléchargement

In [ ]:
# Sauvegarde du checkpoint
ckpt = {
    'epoch'        : NUM_EPOCHS,
    'G_state_dict' : G.state_dict(),
    'D_state_dict' : D.state_dict(),
    'optimizer_G'  : optimizer_G.state_dict(),
    'optimizer_D'  : optimizer_D.state_dict(),
    'losses_G'     : losses_G,
    'losses_D'     : losses_D,
}
ckpt_path = os.path.join(OUTPUT_DIR, 'dcgan_cifar10_final.pt')
torch.save(ckpt, ckpt_path)
print(f'Checkpoint : {ckpt_path}')

# Telecharger les fichiers depuis Colab vers ton Mac
from google.colab import files
files.download(ckpt_path)
files.download(os.path.join(OUTPUT_DIR, 'progression.png'))
files.download(os.path.join(OUTPUT_DIR, 'losses.png'))